# Camada Bronze — Landing_to_Bronze

Carrega os cinco CSVs e o arquivo cotacao_dolar.json do Volume em tabelas Delta.
Cada carga recebe ingestion_datetime e usa Append.


## Configuração da camada Bronze

Usa o mesmo catálogo, schemas e caminho de entrada definidos na preparação do Lakehouse.


In [0]:
# Mantém os caminhos iguais aos do notebook de preparação.

catalog = "cinedata_medallion"

bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")


DataFrame[]

## Carga dos CSVs — Tabelas Bronze

Lê os CSVs com cabeçalho e mantém os campos como texto. Grava tb_movies_info, tb_movies_financials, tb_movies_metrics, tb_credits_and_tags e tb_movies_reviews em Delta, com o timestamp da carga. Append acrescenta registros a cada execução.


In [0]:
# A limpeza e a conversão dos dados serão realizadas na camada Silver.

from pyspark.sql.functions import current_timestamp

# 1. Mapeamento dos caminhos dos arquivos no Volume
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

# 2. Opções de leitura: manter as colunas como texto na Bronze
# Este arquivo contém sinopses com quebras de linha dentro de campos entre aspas.
# O escape de aspas duplicadas segue o formato dos CSVs fornecidos.
df_movies_info_raw = (
    spark.read.option("multiLine", True).option("escape", '"')
    .csv(path_movies_info, header=True, inferSchema=False)
)
df_movies_financials_raw = spark.read.csv(path_movies_financials, header=True, inferSchema=False)
# Os CSVs de métricas e créditos têm aspas corrompidas e column shift na própria origem.
# Mantém-se a leitura por linha para uma aspa quebrada não consumir filmes seguintes.
# Não se tenta reconstruir ou limpar essas colunas na Bronze.
df_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True, inferSchema=False)
df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=False)
df_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True, inferSchema=False)

# 3. Gravação em Delta com timestamp de ingestão
df_movies_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_info")

df_movies_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_financials")

df_movies_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_metrics")

df_credits_and_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_credits_and_tags")

df_movies_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_reviews")


## Cotação do dólar — Carga direta do JSON

Lê cotacao_dolar.json da Landing, desmembra a lista value e grava bronze.tb_cotacao_dolar.
Mantém as colunas de texto usadas na Bronze; a conversão dos tipos é realizada na Silver.
O arquivo fornecido contém cotações de 01 a 07/10/2026.


In [0]:
from pyspark.sql.functions import current_timestamp

# Lê diretamente a resposta salva no Volume.
df_cotacao_dolar_raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{landing_path}/cotacao_dolar.json")
    .selectExpr("explode(value) AS cotacao")
    .selectExpr(
        "CAST(cotacao.dataHoraCotacao AS STRING) AS dataHoraCotacao",
        "CAST(cotacao.cotacaoCompra AS STRING) AS cotacaoCompra"
    )
)

if df_cotacao_dolar_raw.count() == 0:
    raise ValueError("O JSON não contém registros de cotação.")

# Acrescenta os registros na Bronze, com o momento da ingestão.
df_cotacao_dolar_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

display(df_cotacao_dolar_raw)


dataHoraCotacao,cotacaoCompra
2026-10-01 13:10:35.4469,5.2073
2026-10-02 13:03:16.256632,5.2232
2026-10-05 13:07:36.558602,4.9853
2026-10-06 13:03:21.267287,4.9692
2026-10-07 13:05:23.249063,4.9929
